# Belief-Agent v2: Reflection in Action

Demonstrates structured reflection loops with self-critique.

In [ ]:
import json, sys
sys.path.insert(0, "..")
from belief_agent import BeliefAgent, BeliefState
from belief_agent.reflection import reflect, auto_update, human_review

## Setup: an agent with several beliefs

In [ ]:
def critic_llm(messages):
    return json.dumps({
        "critique": "This belief needs more evidence.",
        "new_evidence": ["Recent paper confirms"],
        "new_contradictions": ["Counterexample exists"],
        "updated_confidence": 0.5,
    })

agent = BeliefAgent(llm_call=critic_llm, name="ReflectiveAgent")
agent.adopt(BeliefState("AGI will arrive by 2030", confidence=0.8, evidence=["Trends"]))
agent.adopt(BeliefState("Python is the best for AI", confidence=0.9, evidence=["Ecosystem"]))
agent.adopt(BeliefState("Quantum computing is near", confidence=0.3, evidence=["Hype"]))

## Run structured reflection

In [ ]:
results = reflect(agent, critic_llm, depth=1)
for r in results:
    print(f"{r.proposition}: {r.original_confidence:.2f} -> {r.updated_confidence:.2f} (accepted={r.accepted})")
    print(f"  Critique: {r.critique}")
    print()

## Auto-reflect on low-confidence only

In [ ]:
agent2 = BeliefAgent(llm_call=critic_llm, name="AutoReflect")
agent2.adopt(BeliefState("High", confidence=0.9))
agent2.adopt(BeliefState("Low", confidence=0.2))
results = auto_update(agent2, critic_llm, threshold=0.5)
print(f"Reflected on {len(results)} belief(s)")
for r in results:
    print(f"  - {r.proposition}: {r.original_confidence:.2f} -> {r.updated_confidence:.2f}")

## Human-in-the-loop

In [ ]:
# Simulate a human rejecting the reflection
r = results[0]
r = human_review(r, "Actually, I think the confidence should stay high.", accepted=False)
print(f"After human review: accepted={r.accepted}")
print(f"Critique now: {r.critique[:80]}...")